# Phase 3 標籤工程測試與驗證（Multi-timeframe）

本 Notebook 用於：
- 讀取每種 label 的 checkpoint（5 種）
- 檢查欄位命名與 dtype（+1/-1）
- 檢查 NaN 比率與正負比例
- **驗證與 Phase 2 combined_features 時間索引完全對齊**
- 簡單對比圖（截取一小段）

資料由 FeaturePaths / LabelPaths 與 DATA_LAKE_ROOT 解析（預設 C:\Data_Lake）


In [ ]:
import os\nimport sys\nfrom pathlib import Path\n\nimport pandas as pd\nimport matplotlib.pyplot as plt\n\nROOT = Path(os.getcwd()).resolve().parent\nif str(ROOT) not in sys.path:\n    sys.path.insert(0, str(ROOT))\n\nprint('Project root =', ROOT)\nprint('DATA_LAKE_ROOT =', os.environ.get('DATA_LAKE_ROOT', '(未設定，將使用 utils/config.py 預設)'))

In [ ]:
from features.feature_utils import FEATURES_PER_SYMBOL, FeaturePaths, WINDOWS
from labels.label_utils import LabelPaths, load_label, load_raw_klines_for_labels
from utils.config import SYMBOLS
import pandas as pd

paths = LabelPaths.default()
feat_paths = FeaturePaths.default()
print('Labels root =', paths.root)
print('Features root =', feat_paths.root)
print('Symbols =', SYMBOLS)
print('Phase 2 WINDOWS =', WINDOWS, 'FEATURES_PER_SYMBOL =', FEATURES_PER_SYMBOL)

LABELS = [
    ('realized_volatility', 'label_realized_volatility'),
    ('sequential_correlation', 'label_sequential_correlation'),
    ('skewness', 'label_skewness'),
    ('kurtosis', 'label_kurtosis'),
    ('jarque_bera', 'label_jarque_bera'),
]


In [ ]:
def pos_neg_ratio(s: pd.Series):
    v = s.dropna()
    if len(v) == 0:
        return 0.0, 0.0
    return float((v==1).mean()), float((v==-1).mean())

for sym in SYMBOLS:
    print('\n====', sym, '====')
    for label_type, col in LABELS:
        df = load_label(sym, label_type)
        assert col in df.columns, f'{label_type} missing {col}'
        s = df[col]
        pr, nr = pos_neg_ratio(s)
        print(f'OK {label_type}: dtype={s.dtype}, nan={s.isna().mean():.2%}, pos/neg={pr:.2%}/{nr:.2%}')


## Phase 2 / Phase 3 索引對齊驗證

In [ ]:
for sym in SYMBOLS:
    feat_p = feat_paths.combined_path(sym)
    lab_p = paths.combined_path(sym)
    if not feat_p.is_file():
        print(f'FAIL {sym}: missing Phase 2 {feat_p}')
        continue
    if not lab_p.is_file():
        print(f'FAIL {sym}: missing Phase 3 {lab_p}')
        continue
    feat = pd.read_parquet(feat_p)
    lab = pd.read_parquet(lab_p)
    match = feat.index.equals(lab.index)
    print(
        f'{"OK" if match else "FAIL"} {sym}: '
        f'feat_cols={len(feat.columns)} lab_cols={len(lab.columns)} '
        f'len_feat={len(feat)} len_lab={len(lab)} index_match={match}'
    )
    if match:
        print('  first 3:', feat.index[:3].tolist())
        print('  last 3:', feat.index[-3:].tolist())


## 簡單對比圖（截取最後 7 天）

In [ ]:
sym = SYMBOLS[0]
raw = load_raw_klines_for_labels(sym)
labels = load_label(sym, 'realized_volatility')
tail_close = raw[['close']].tail(60*24*7)
tail_lab = labels.tail(60*24*7)
fig, ax1 = plt.subplots(figsize=(12,4))
ax1.plot(tail_close.index, tail_close['close'], label='close')
ax1.set_title(f'{sym} close vs label_realized_volatility (last 7 days)')
ax1.set_ylabel('close')
ax2 = ax1.twinx()
ax2.plot(tail_lab.index, tail_lab['label_realized_volatility'].astype('float'), color='orange', alpha=0.5)
plt.show()
